# SQL Engineering & Business Analysis


---

## Marketing Analytics Engineering Platform (V2)

### Project Context

This notebook documents the SQL-based business analysis component of the Marketing Analytics Engineering Platform (V2). It connects analytical SQL implementation with business interpretation, using campaign performance data to examine marketing effectiveness, advertising efficiency, and performance differences across campaign characteristics.

The notebook complements `analysis/business_analysis.sql` by explaining the analytical approach, presenting verified query results, interpreting findings, and documenting relevant assumptions and limitations.

**My stack:** SQL, Python, PostgreSQL, and Jupyter Notebook.

## 1.1 Business Purpose

Marketing teams need to understand how effectively their campaigns turn advertising spend and marketing activity into measurable business results. Evaluating campaigns by revenue or volume alone can be misleading because campaigns may differ in spending levels, audience reach, and efficiency.

This analysis uses SQL to evaluate campaign performance across marketing channels and campaign characteristics. It examines aggregate performance, revenue generation, return on advertising spend (ROAS), and marketing efficiency to identify performance patterns and areas that warrant further investigation.

The goal is to translate campaign-level data into actionable business insights through transparent SQL analysis, supported by Python-based execution and validation where appropriate.

## 1.2 Analytical Objectives

This analysis aims to:

1. **Establish an overall performance baseline** using campaign counts, impressions, clicks, conversions, advertising spend, revenue, click-through rate (CTR), and ROAS.
2. **Compare marketing channels** using consistent aggregate metrics to identify differences in performance and efficiency.
3. **Identify leading campaigns** by revenue and ROAS, using a minimum-spend threshold for more meaningful ROAS comparisons.
4. **Examine performance distributions** using percentiles and a defined low-ROAS screening criterion.
5. **Evaluate marketing efficiency** through cost per click (CPC), cost per thousand impressions (CPM), ROI ratio, and ROAS.
6. **Compare campaign characteristics** across devices, product categories, regions, and target audiences.
7. **Develop evidence-based investigation priorities** from the results while distinguishing observed patterns from causal conclusions.

The analysis is descriptive. It identifies patterns and potential areas for investigation but does not independently establish why a campaign performed as it did or prove that a particular marketing change will improve future results.

## 1.3 Dataset Context and Scope

The analysis uses three CSV source files containing marketing campaign performance records:

- `marketing_campaign_2024.csv`
- `marketing_campaign_2025.csv`
- `marketing_campaign_jul_dec_2024.csv`

Together, the files contain **1,831 campaign records** and **15 columns** covering campaign identifiers, campaign dates, marketing channels, regions, audience and product characteristics, device types, impressions, clicks, conversions, advertising spend, and revenue.

The analysis focuses on campaign-level performance. It aggregates measures across campaigns and compares results by channel and selected campaign characteristics. The SQL analysis references the following PostgreSQL objects:

- `analytics.campaign_metrics`
- `analytics.dim_campaign`
- `analytics.fct_campaign_performance`

**Scope boundaries and data considerations**

- The analysis is descriptive and does not establish causal relationships between campaign characteristics and performance.
- ROAS measures attributed revenue relative to advertising spend; it is not equivalent to net profit or a complete measure of marketing profitability.
- Low-ROAS screening uses a defined analytical threshold and minimum-spend criterion. These are investigation criteria, not automatically approved business rules.
- The upstream transformation reconstructs the impressions, clicks, and conversions measures by ordering source values. Although this transformation has been unit-tested, the semantic correctness of the reconstructed measures has not been independently established.
- Database object availability and the current execution workflow must be verified before results are presented as freshly executed and reproducible.

These boundaries will guide the interpretation of results throughout the notebook.

---


# 2. Environment & Data Access

This section documents the software environment, database dependencies, and execution workflow required to run the business analysis.

The notebook combines SQL for analytical queries, Python for database access and result inspection, and Markdown for documenting the methodology and interpreting findings.

The analysis queries in `analysis/business_analysis.sql` reference PostgreSQL objects, while the project's dbt development environment is configured for DuckDB. These databases serve different roles in the workflow and must not be treated as interchangeable without verification.

The following subsections establish the required dependencies, verify database access, and document how analytical queries are executed and their results reproduced.

In [1]:
import sys
import duckdb
import pandas as pd
import psycopg
import IPython
import yaml

from pathlib import Path
import importlib.util

In [2]:
# check python

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\JayPogi\Documents\PY\Py org IDLE\python.exe
Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]


In [3]:
# tools and libraries check

packages = ["duckdb", "pandas", "psycopg", "psycopg2", "sqlalchemy", "ipykernel"]

for package in packages:
    print(f"{package}: {'available' if importlib.util.find_spec(package) else 'not installed'}")

duckdb: available
pandas: available
psycopg: available
psycopg2: not installed
sqlalchemy: not installed
ipykernel: available


In [4]:
# version check for installed packages
print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("DuckDB version:", duckdb.__version__)
print("pandas version:", pd.__version__)
print("psycopg version:", psycopg.__version__)
print("IPython version:", IPython.__version__)

Python version: 3.13.3
Python executable: c:\Users\JayPogi\Documents\PY\Py org IDLE\python.exe
DuckDB version: 1.5.5
pandas version: 2.3.1
psycopg version: 3.3.3
IPython version: 9.4.0


In [5]:
# connection test

print(f"psycopg version: {psycopg.__version__}")
print("PostgreSQL driver is available.")

psycopg version: 3.3.3
PostgreSQL driver is available.


In [ ]:
# database configuration

project_root = Path.cwd()

print("Current directory:", project_root)
print("SQL file exists:", (project_root / "analysis" / "business_analysis.sql").exists())
print("dbt project exists:", (project_root / "dbt_marketing_analytics_engineering").exists())


In [ ]:
# project root??

current_dir = Path.cwd()
project_root = next(
    (parent for parent in [current_dir, *current_dir.parents]
     if (parent / "analysis" / "business_analysis.sql").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")

sql_file = project_root / "analysis" / "business_analysis.sql"

print("Project root:", project_root)
print("SQL file exists:", sql_file.exists())
print("SQL file size:", sql_file.stat().st_size, "bytes")




sql_text = sql_file.read_text(encoding="utf-8")

print(sql_text[:6000])



In [8]:

sql_text = sql_file.read_text(encoding="utf-8")

print(f"Loaded business_analysis.sql ({len(sql_text):,} characters)")
print(sql_text)

Loaded business_analysis.sql (16,692 characters)
/*
Project: Marketing Analytics Engineering Platform (V2)



SQL Engineering & Business Analysis
File: business_analysis.sql

Purpose:
Document SQL analysis supporting marketing campaign performance
evaluation and business recommendations.

Database:
PostgreSQL
Schema:
analytics

Serving tables:
- analytics.dim_campaign
- analytics.fct_campaign_performance
- analytics.campaign_metrics (computed metrics view)

Analytical conventions:
- Campaign grain: one row per campaign.
- Performance metrics are aggregated from campaign-level measures.
- Ratios are calculated from aggregated numerators and denominators
  where appropriate.
- ROAS measures revenue relative to advertising spend.
- ROI is calculated as (revenue - spend) / spend.
- Cost per conversion is used instead of CPA where the business
  meaning of "conversion" has not been independently verified.
- Findings are descriptive and do not establish causation.
*/


/* ===================

### 2.1 Python Environment and Dependencies

The notebook runs on Python 3.13.3 using the standalone Python interpreter configured in the local development environment.

The following library versions were verified during environment inspection:

| Component | Version | Purpose |
|---|---|---|
| Python | 3.13.3 | Notebook execution |
| DuckDB | 1.5.5 | Local analytical database operations |
| pandas | 2.3.1 | Tabular result inspection and analysis |
| psycopg | 3.3.3 | PostgreSQL database connectivity |
| IPython | 9.4.0 | Interactive notebook execution |

These versions describe the environment used during inspection. A reproducible setup should document the required dependencies and compatible versions separately.

The notebook currently uses a different interpreter from the project's `.venv`. Consequently, dependencies available in one environment should not be assumed to be available in the other.

### 2.2 SQL Execution Workflow

The business analysis is implemented in `analysis/business_analysis.sql`, which serves as the reference for the analytical queries and their implemented logic.

The notebook complements this SQL file by documenting the analytical approach, presenting results where available, and interpreting findings using Python and pandas where appropriate.

The workflow separates analytical implementation from presentation:

1. **SQL analysis:** Calculate aggregate performance metrics, compare marketing channels, rank campaigns, examine distributions, and evaluate campaign characteristics.
2. **Python integration:** Retrieve and inspect query results, organize tabular outputs, and produce visualizations where they improve interpretation.
3. **Business interpretation:** Explain observed patterns, identify areas for further investigation, and document assumptions and limitations.

The analysis references PostgreSQL serving-layer objects. Execution and result verification are treated separately from the documented query logic; the notebook does not assume that the database is automatically refreshed whenever the dbt models are built.

---

# 3. Overall Performance

## 3.1 Performance Baseline

The overall performance baseline establishes a consolidated view of campaign activity across the dataset. It summarizes campaign volume, audience engagement, conversions, advertising spend, and revenue before examining individual marketing channels or campaign characteristics.

The analysis uses aggregate metrics to establish a consistent reference point for the subsequent comparisons. Click-through rate (CTR) is calculated from total clicks divided by total impressions, while return on advertising spend (ROAS) is calculated from total revenue divided by total advertising spend.

These aggregate ratios provide a more representative overall baseline than simply averaging campaign-level percentages.

## 3.2 Baseline KPI Results

The baseline query consolidates campaign-level measures into a single summary of overall marketing performance.

The summary includes:

- **Campaign volume:** Number of campaign records included in the analysis.
- **Impressions and clicks:** Measures of campaign reach and engagement activity.
- **Conversions:** Recorded conversion volume across the campaigns.
- **Advertising spend and revenue:** The total recorded expenditure and attributed revenue.
- **Click-through rate (CTR):** Total clicks divided by total impressions.
- **Return on advertising spend (ROAS):** Total attributed revenue divided by total advertising spend.

These metrics establish the reference point for the channel comparisons and campaign-level investigations that follow. Their interpretation depends on the validity of the underlying measures and the consistency of the reporting scope.

## 3.3 SQL Implementation

The baseline performance query aggregates campaign records into a single summary row. It uses aggregate functions to calculate total campaign volume, impressions, clicks, conversions, advertising spend, and revenue.

The query also derives CTR and ROAS from the aggregated measures. Calculating these ratios from totals ensures that the overall metrics reflect the combined dataset rather than giving every campaign percentage equal weight.

The SQL implementation is maintained in `analysis/business_analysis.sql` to keep the notebook aligned with the project's analytical query source.

In [9]:
# DB

db_files = list(project_root.rglob("*.duckdb"))

if db_files:
    print("DuckDB files found:")
    for db_file in db_files:
        print(db_file.relative_to(project_root))
else:
    print("No .duckdb files found inside the project directory.")


DuckDB files found:
marketing_analytics.duckdb


In [10]:
# Connect to DuckDB and list tables 

db_path = project_root / "marketing_analytics.duckdb"

conn = duckdb.connect(str(db_path), read_only=True)

tables_df = conn.execute("""
    SELECT
        table_schema,
        table_name,
        table_type
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").fetchdf()

display(tables_df)


,table_schema,table_name,table_type
0,main,campaign_metrics,VIEW
1,main,dim_campaign,VIEW
2,main,fct_campaign_performance,VIEW
3,main,int_campaign_performance,VIEW
4,main,stg_campaign_performance,VIEW
5,raw,marketing_campaign,BASE TABLE


In [11]:
# Overall campaign performance baseline

baseline_df = conn.execute("""
    SELECT
        COUNT(*) AS campaign_rows,
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        SUM(conversions) AS total_conversions,
        ROUND(SUM(spend_usd), 2) AS total_spend_usd,
        ROUND(SUM(revenue_usd), 2) AS total_revenue_usd,
        ROUND(
            SUM(clicks)::NUMERIC
            / NULLIF(SUM(impressions), 0) * 100,
            2
        ) AS aggregate_ctr_pct,
        ROUND(
            SUM(revenue_usd)::NUMERIC
            / NULLIF(SUM(spend_usd), 0),
            2
        ) AS aggregate_roas
    FROM main.fct_campaign_performance
""").fetchdf()

display(baseline_df)


,campaign_rows,total_impressions,total_clicks,total_conversions,total_spend_usd,total_revenue_usd,aggregate_ctr_pct,aggregate_roas
0,1831,156643739.0,52352778.0,23716663.0,47118927.37,1.267274e+08,33.42,2.69


In [12]:

display(conn.execute("""
    SELECT *
    FROM main.fct_campaign_performance
    LIMIT 5
""").fetchdf())

print("Fact table row count:")
display(conn.execute("""
    SELECT COUNT(*) AS row_count
    FROM main.fct_campaign_performance
""").fetchdf())


,campaign_id,impressions,clicks,conversions,spend_usd,revenue_usd
0,2024_0001,65466,28252,5609,39193.43,79017.74
1,2024_0002,89608,83584,26865,17291.53,49868.54
2,2024_0003,62661,43662,37853,6729.63,63021.28
3,2024_0004,75023,41421,10577,15077.58,133106.71
4,2024_0005,84039,56010,11283,16877.69,144736.99


Fact table row count:


,row_count
0,1831


In [13]:
# baseline query


baseline_df = conn.execute("""
    SELECT
        COUNT(*) AS campaign_rows,
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        SUM(conversions) AS total_conversions,
        ROUND(SUM(spend_usd), 2) AS total_spend_usd,
        ROUND(SUM(revenue_usd), 2) AS total_revenue_usd,
        ROUND(
            100.0 * SUM(clicks) / NULLIF(SUM(impressions), 0),
            2
        ) AS aggregate_ctr_pct,
        ROUND(
            SUM(revenue_usd) / NULLIF(SUM(spend_usd), 0),
            2
        ) AS aggregate_roas
    FROM main.fct_campaign_performance
""").fetchdf()

display(baseline_df.T.rename(columns={0: "value"}))


,value
campaign_rows,1.831000e+03
total_impressions,1.566437e+08
total_clicks,5.235278e+07
total_conversions,2.371666e+07
total_spend_usd,4.711893e+07
total_revenue_usd,1.267274e+08
aggregate_ctr_pct,3.342000e+01
aggregate_roas,2.690000e+00


### 3.4 KPI Calculation Definitions

The analysis derives performance and efficiency metrics from aggregated campaign measures. Ratio-based metrics are calculated from their underlying totals where appropriate, rather than averaging individual campaign ratios.

| KPI | Calculation | Interpretation |
|---|---|---|
| Click-Through Rate (CTR) | Total Clicks / Total Impressions | Share of impressions that generated clicks |
| Cost per Click (CPC) | Total Spend / Total Clicks | Average advertising cost per click |
| Cost per Mille (CPM) | Total Spend / Total Impressions × 1,000 | Advertising cost per 1,000 impressions |
| Return on Advertising Spend (ROAS) | Total Revenue / Total Spend | Revenue generated per unit of advertising spend |
| ROI Ratio | (Total Revenue − Total Spend) / Total Spend | Return relative to advertising spend, excluding other business costs |

**Calculation considerations:**

- Ratio metrics should handle zero denominators safely, returning `NULL` or an equivalent missing value when a ratio cannot be calculated.
- ROAS and ROI are related but distinct. ROAS measures revenue relative to ad spend, while the ROI ratio measures revenue remaining after ad spend relative to that spend.
- These metrics depend on the accuracy and business meaning of the underlying impressions, clicks, conversions, spend, and revenue measures.
- The ROI ratio used here is an advertising-spend-only measure, not a complete net-profit calculation.
- Campaign-level ratios should not be averaged indiscriminately to represent overall performance. Use the appropriate aggregated numerator and denominator for portfolio-level KPIs.

---

# 4. Channel Performance

## 4.1 Analytical Objective

This section compares marketing channel performance to identify differences in engagement, conversion activity, advertising expenditure, and attributed revenue.

The analysis groups campaign records by channel and aggregates their underlying measures. This provides a consistent basis for comparing channel-level performance while preserving the distinction between activity volume, cost efficiency, and revenue outcomes.

## 4.2 Comparison Metrics

The channel analysis evaluates the following measures:

- **Campaign volume:** Number of campaigns associated with each channel.
- **Impressions and clicks:** Indicators of campaign exposure and engagement activity.
- **Conversions:** Recorded conversion volume.
- **Advertising spend and revenue:** Total expenditure and attributed revenue by channel.
- **CTR:** Click-through rate calculated from total clicks divided by total impressions.
- **ROAS:** Return on advertising spend calculated from total revenue divided by total advertising spend.

Where applicable, CPC and CPM provide additional context on advertising efficiency.


In [14]:

# CHANNEL PERFORMANCE

channel_df = conn.execute("""
    SELECT
        d.channel,
        COUNT(*) AS campaign_count,
        SUM(f.impressions) AS total_impressions,
        SUM(f.clicks) AS total_clicks,
        SUM(f.conversions) AS total_conversions,
        ROUND(SUM(f.spend_usd), 2) AS total_spend_usd,
        ROUND(SUM(f.revenue_usd), 2) AS total_revenue_usd,
        ROUND(
            100.0 * SUM(f.clicks)
            / NULLIF(SUM(f.impressions), 0), 2
        ) AS ctr_pct,
        ROUND(
            SUM(f.spend_usd)
            / NULLIF(SUM(f.conversions), 0), 2
        ) AS cost_per_conversion_usd,
        ROUND(
            SUM(f.revenue_usd)
            / NULLIF(SUM(f.spend_usd), 0), 2
        ) AS roas
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    GROUP BY d.channel
    ORDER BY roas DESC, total_revenue_usd DESC
""").fetchdf()

display(channel_df)


,channel,campaign_count,total_impressions,total_clicks,total_conversions,total_spend_usd,total_revenue_usd,ctr_pct,cost_per_conversion_usd,roas
0,Search,349,29833001.0,10804653.0,5145215.0,9192231.70,25250215.62,36.22,1.79,2.75
1,Social,358,30920081.0,10302672.0,4589995.0,8627773.36,23708663.24,33.32,1.88,2.75
2,Print,386,33180671.0,11525944.0,5171693.0,9863377.27,26417035.40,34.74,1.91,2.68
3,Email,389,33681945.0,11238251.0,5301071.0,10470238.67,27959137.11,33.37,1.98,2.67
4,Display,349,29028041.0,8481258.0,3508689.0,8965306.37,23392376.46,29.22,2.56,2.61


In [15]:

print("Columns:", channel_df.columns.tolist())
display(channel_df[[
    "channel",
    "campaign_count",
    "total_impressions",
    "total_clicks",
    "total_conversions",
    "total_spend_usd",
    "total_revenue_usd",
    "ctr_pct",
    "cost_per_conversion_usd",
    "roas"
]].to_string(index=False))


Columns: ['channel', 'campaign_count', 'total_impressions', 'total_clicks', 'total_conversions', 'total_spend_usd', 'total_revenue_usd', 'ctr_pct', 'cost_per_conversion_usd', 'roas']


'channel  campaign_count  total_impressions  total_clicks  total_conversions  total_spend_usd  total_revenue_usd  ctr_pct  cost_per_conversion_usd  roas\n Search             349         29833001.0    10804653.0          5145215.0       9192231.70        25250215.62    36.22                     1.79  2.75\n Social             358         30920081.0    10302672.0          4589995.0       8627773.36        23708663.24    33.32                     1.88  2.75\n  Print             386         33180671.0    11525944.0          5171693.0       9863377.27        26417035.40    34.74                     1.91  2.68\n  Email             389         33681945.0    11238251.0          5301071.0      10470238.67        27959137.11    33.37                     1.98  2.67\nDisplay             349         29028041.0     8481258.0          3508689.0       8965306.37        23392376.46    29.22                     2.56  2.61'

## 4.4 Results and Interpretation

The channel-level aggregation shows differences in campaign volume, advertising expenditure, and attributed revenue.

- **Email** has the highest campaign count (389) and attributed revenue ($10.47 million), but also the highest total spend ($5.30 million).
- **Search** and **Social** have the highest ROAS at 2.75x, indicating stronger revenue relative to advertising spend in this dataset.
- **Display** has the lowest CTR (29.22%) and ROAS (2.61x), alongside the highest cost per conversion ($2.56).
- **Print** records the highest conversion volume (5.17 million), while Email records the highest total impressions (33.68 million).

These results describe observed differences across channels. They do not establish that channel choice alone caused the differences, and should be interpreted alongside the data-quality limitations documented in this project.

---

# 5. Campaign Leaders

## 5.1 Analytical Objective

This section identifies leading campaigns using two complementary performance measures: attributed revenue and return on advertising spend (ROAS).

Revenue highlights campaigns that contribute the largest absolute amount of attributed sales, while ROAS evaluates revenue generated relative to advertising expenditure. Using both measures provides a more balanced view of campaign performance.

## 5.2 Top Campaigns by Revenue

Campaigns are ranked by attributed revenue to identify the largest revenue contributors in the dataset.

This ranking helps identify campaigns that account for a substantial share of recorded revenue. However, high revenue alone does not establish that a campaign is efficient or profitable because campaigns may differ considerably in advertising expenditure.

## 5.3 Top Campaigns by ROAS

Campaigns are also ranked by ROAS, calculated as attributed revenue divided by advertising spend.

To improve comparability, the implemented analysis applies a minimum advertising-spend threshold of **$10,000**. This limits the ranking to campaigns that meet the selected expenditure criterion and reduces the influence of very small-spend campaigns on the comparison.

The threshold is an analytical screening choice for this project, not an established universal standard or an approved business rule.


In [16]:
# 5.1 TOP CAMPAIGNS BY REVENUE

top_revenue_df = conn.execute("""
    SELECT
        d.campaign_id,
        d.campaign_name,
        d.channel,
        d.region,
        d.product_category,
        ROUND(f.spend_usd, 2) AS spend_usd,
        ROUND(f.revenue_usd, 2) AS revenue_usd,
        ROUND(m.ctr * 100, 2) AS ctr_pct,
        ROUND(m.cpa, 2) AS cost_per_conversion_usd,
        ROUND(m.roas, 2) AS roas
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    JOIN main.campaign_metrics AS m
        ON f.campaign_id = m.campaign_id
    ORDER BY f.revenue_usd DESC, d.campaign_id
    LIMIT 10
""").fetchdf()

display(top_revenue_df)


,campaign_id,campaign_name,channel,region,product_category,spend_usd,revenue_usd,ctr_pct,cost_per_conversion_usd,roas
0,2024_JD_0220,Campaign_2024_JD_0220,Search,North America,Automotive,45679.97,172113.15,7.89,60.03,3.77
1,2024_JD_0236,Campaign_2024_JD_0236,Display,South America,Electronics,49820.36,171482.89,4.13,1132.28,3.44
2,2024_JD_0290,Campaign_2024_JD_0290,Search,South America,Clothing,49631.69,165377.86,9.05,10.13,3.33
3,2024_JD_0484,Campaign_2024_JD_0484,Email,Europe,Automotive,47931.70,161446.71,3.29,90.10,3.37
4,2024_JD_0496,Campaign_2024_JD_0496,Print,South America,Clothing,47241.38,161013.36,12.99,289.82,3.41
5,2024_JD_0126,Campaign_2024_JD_0126,Social,South America,Health,46546.93,159258.79,10.95,37.84,3.42
6,2024_JD_0181,Campaign_2024_JD_0181,Email,Asia,Home,46295.41,158952.23,3.92,49.36,3.43
7,2024_JD_0789,Campaign_2024_JD_0789,Email,Asia,Home,46295.41,158952.23,3.92,49.36,3.43
8,2024_JD_0790,Campaign_2024_JD_0790,Email,Asia,Home,46295.41,158952.23,3.92,49.36,3.43
9,2024_JD_0557,Campaign_2024_JD_0557,Print,North America,Home,43394.47,157764.96,3.26,374.09,3.64


In [17]:

display(conn.execute("""
    SELECT
        campaign_id,
        impressions,
        clicks,
        conversions,
        spend_usd,
        revenue_usd
    FROM main.fct_campaign_performance
    WHERE campaign_id IN (
        '2024_JD_0220',
        '2024_JD_0236',
        '2024_JD_0290',
        '2024_JD_0484',
        '2024_JD_0496',
        '2024_JD_0126',
        '2024_JD_0181',
        '2024_JD_0789',
        '2024_JD_0790',
        '2024_JD_0557'
    )
    ORDER BY revenue_usd DESC
""").fetchdf())


,campaign_id,impressions,clicks,conversions,spend_usd,revenue_usd
0,2024_JD_0220,87879,6937,761,45679.97,172113.15
1,2024_JD_0236,21082,871,44,49820.36,171482.89
2,2024_JD_0290,191994,17382,4898,49631.69,165377.86
3,2024_JD_0484,54067,1781,532,47931.70,161446.71
4,2024_JD_0496,30372,3945,163,47241.38,161013.36
5,2024_JD_0126,102829,11258,1230,46546.93,159258.79
6,2024_JD_0181,144892,5680,938,46295.41,158952.23
7,2024_JD_0789,144892,5680,938,46295.41,158952.23
8,2024_JD_0790,144892,5680,938,46295.41,158952.23
9,2024_JD_0557,13024,424,116,43394.47,157764.96


### 5.2.1 Results and Interpretation

The revenue ranking identifies campaigns with the largest attributed revenue in the dataset. Campaign 2024_JD_0220 ranks first, generating $172,113.15 in revenue from $45,679.97 in advertising spend, with a reported ROAS of 3.77x.

Campaign 2024_JD_0236 ranks second by revenue at $171,482.89, but records only 44 conversions. Its comparatively high cost per conversion indicates that revenue ranking alone does not provide a complete picture of campaign efficiency.

Three campaigns—2024_JD_0181, 2024_JD_0789, and 2024_JD_0790—have identical recorded performance measures despite having different campaign IDs. These records are retained as separate campaigns pending further investigation.

---

In [18]:
# 5.3 TOP CAMPAIGNS BY ROAS

# Minimum advertising spend: $10,000
top_roas_df = conn.execute("""
    SELECT
        d.campaign_id,
        d.campaign_name,
        d.channel,
        d.region,
        ROUND(f.spend_usd, 2) AS spend_usd,
        ROUND(f.revenue_usd, 2) AS revenue_usd,
        ROUND(m.roas, 2) AS roas,
        ROUND(m.cpa, 2) AS cost_per_conversion_usd,
        ROUND(m.ctr * 100, 2) AS ctr_pct
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    JOIN main.campaign_metrics AS m
        ON f.campaign_id = m.campaign_id
    WHERE f.spend_usd >= 10000
    ORDER BY
        m.roas DESC,
        f.revenue_usd DESC,
        d.campaign_id
    LIMIT 10
""").fetchdf()

display(top_roas_df)


,campaign_id,campaign_name,channel,region,spend_usd,revenue_usd,roas,cost_per_conversion_usd,ctr_pct
0,2025_0046,Campaign_2025_0046,Search,Africa,10213.07,142753.69,13.98,1.14,65.00
1,2024_0374,Campaign_2024_0374,Social,Africa,11166.26,149599.72,13.40,0.49,80.02
2,2024_0455,Campaign_2024_0455,Email,South America,10663.65,142336.50,13.35,0.34,78.87
3,2025_0137,Campaign_2025_0137,Social,Europe,11665.11,149569.44,12.82,0.57,67.01
4,2025_0442,Campaign_2025_0442,Display,North America,10335.44,129159.37,12.50,2.32,70.85
5,2025_0181,Campaign_2025_0181,Search,Europe,11513.21,143470.26,12.46,0.52,85.08
6,2025_0277,Campaign_2025_0277,Print,South America,12425.23,149713.74,12.05,0.79,60.08
7,2024_0375,Campaign_2024_0375,Email,South America,10605.03,123435.17,11.64,2.31,37.58
8,2024_0021,Campaign_2024_0021,Search,Asia,10585.37,121133.22,11.44,0.83,79.86
9,2025_0399,Campaign_2025_0399,Social,Europe,10167.74,116070.80,11.42,0.94,51.46


### 5.3.1 Results and Interpretation

The top-ranked campaign is 2025_0046, with $142,753.69 in attributed revenue from $10,213.07 in advertising spend, producing a reported ROAS of 13.98x. All ten campaigns exceed the $10,000 spend threshold, and their reported ROAS ranges from 11.42x to 13.98x.

These rankings identify campaigns with high attributed revenue relative to advertising spend within the selected threshold. The results should be validated against the underlying measures before being used to make budget-allocation decisions.

---

# 6. Performance Distribution

## 6.1 Analytical Objective

This section examines the distribution of campaign-level return on advertising spend (ROAS) to understand differences in performance across campaigns.

While aggregate metrics summarize overall results, they can conceal substantial variation between individual campaigns. Distribution-based analysis provides additional context by showing how campaigns perform relative to one another and helps establish a systematic approach to identifying campaigns that warrant further investigation.

## 6.2 ROAS Distribution and Performance Benchmarks

The analysis uses percentile-based benchmarks to summarize the distribution of campaign ROAS.

Percentiles describe the relative position of observations within a distribution. For example, the 25th percentile (P25) identifies the value at or below which approximately one-quarter of the observations fall, subject to the percentile calculation method and the presence of tied values.

These benchmarks help characterize campaign performance without assuming that every campaign should achieve an identical return. They are descriptive reference points rather than universal performance standards.

## 6.3 Low-ROAS Campaign Screening

The SQL analysis identifies campaigns with ROAS below **1.66** and advertising spend of at least **$10,000** for further investigation.

The ROAS threshold of 1.66 is a rounded P25-based screening criterion used in this project. It is not a formally approved business rule or a universal definition of underperformance.

Combining the ROAS threshold with a minimum-spend requirement focuses attention on campaigns that meet both the selected return criterion and expenditure threshold.

## 6.4 Channel-Level Investigation

The analysis also examines the channel distribution of campaigns that meet the low-ROAS screening criteria.

This filtered comparison helps identify where screened campaigns occur across channels. Because the analysis includes only campaigns meeting the specified ROAS and spend conditions, its results should not be interpreted as a complete comparison of overall channel performance.



In [19]:
# 6. ROAS DISTRIBUTION AND PERFORMANCE BENCHMARKS

roas_distribution_df = conn.execute("""
    SELECT
        COUNT(*) AS campaign_count,

        COUNT(*) FILTER (
            WHERE spend_usd >= 10000
        ) AS campaigns_spend_10k_plus,

        ROUND(MIN(roas), 2) AS min_roas,

        ROUND(
            PERCENTILE_CONT(0.25)
            WITHIN GROUP (ORDER BY roas), 2
        ) AS p25_roas,

        ROUND(
            PERCENTILE_CONT(0.50)
            WITHIN GROUP (ORDER BY roas), 2
        ) AS median_roas,

        ROUND(
            PERCENTILE_CONT(0.75)
            WITHIN GROUP (ORDER BY roas), 2
        ) AS p75_roas,

        ROUND(MAX(roas), 2) AS max_roas,

        ROUND(
            PERCENTILE_CONT(0.50)
            WITHIN GROUP (ORDER BY ctr * 100), 2
        ) AS median_ctr_pct,

        ROUND(MAX(ctr * 100), 2) AS max_ctr_pct

    FROM main.campaign_metrics
""").fetchdf()

display(roas_distribution_df.T.rename(columns={0: "value"}))


,value
campaign_count,1831.00
campaigns_spend_10k_plus,1520.00
min_roas,0.03
p25_roas,1.66
median_roas,2.59
p75_roas,3.42
max_roas,133.44
median_ctr_pct,28.02
max_ctr_pct,99.97


### 6.1 Results and Interpretation

The campaign-level median ROAS is **2.59x**, with the middle 50% of campaigns ranging from **1.66x (25th percentile)** to **3.42x (75th percentile)**. The minimum observed ROAS is 0.03x, while the maximum is 133.44x, indicating substantial variation across campaigns.

Of the 1,831 campaigns, **1,520 meet the $10,000 minimum-spend screening condition**. This threshold is used to define the eligible campaign set for the ROAS ranking; it is not a universal performance standard.

The median CTR is **28.02%**, and the maximum is 99.97%. These figures describe the observed distribution and should be interpreted alongside the previously identified data-quality concerns.

The median campaign ROAS differs from aggregate ROAS because the median treats each campaign as one observation, whereas aggregate ROAS weights performance by advertising spend.

---

# 7. Marketing Efficiency

## 7.1 Analytical Objective

This section evaluates advertising efficiency by examining the relationship between marketing expenditure, campaign activity, and attributed revenue.

Revenue and conversion volume provide useful measures of campaign output, but they do not explain how much advertising expenditure was required to generate those results. Cost and return metrics provide additional context for assessing the efficiency of marketing activity across channels.

## 7.2 Cost Efficiency Metrics

The analysis uses the following metrics to evaluate advertising costs:

- **Cost per Click (CPC):** Total advertising spend divided by total clicks. This measures the average advertising cost per recorded click.
- **Cost per Mille (CPM):** Total advertising spend divided by total impressions, multiplied by 1,000. This measures the cost of generating 1,000 impressions.

CPC and CPM represent different aspects of advertising efficiency. CPC relates expenditure to engagement activity, while CPM relates expenditure to campaign exposure.

Neither metric independently establishes the quality of the traffic, the value of the resulting conversions, or the overall profitability of a campaign.

## 7.3 Return Metrics

The analysis also examines two return measures:

- **Return on Advertising Spend (ROAS):** Attributed revenue divided by advertising spend.
- **ROI Ratio:** Attributed revenue minus advertising spend, divided by advertising spend.

ROAS measures revenue generated per unit of advertising expenditure. The ROI ratio expresses the difference between attributed revenue and advertising spend relative to that spend.

For example, a ROAS of 2.0 represents $2 in attributed revenue per $1 of advertising spend. Under the project's advertising-spend-only ROI definition, this corresponds to an ROI ratio of 1.0, or 100%.

This ROI calculation excludes other costs, such as product costs, salaries, fulfillment, and overhead. It should not be interpreted as a complete measure of business profit.

## 7.4 Channel-Level Comparison

The SQL analysis compares these efficiency and return metrics across marketing channels using aggregated measures.

The results should be evaluated collectively. A channel with lower CPC may not generate higher revenue, and a channel with higher ROAS may not produce the largest absolute revenue contribution.

Comparisons should also account for differences in campaign scale and the amount of advertising spend represented by each channel. Ratios with zero denominators should be treated as undefined rather than interpreted as valid numerical results.



In [20]:
# 7. LOW-ROAS CAMPAIGNS FOR INVESTIGATION

low_roas_df = conn.execute("""
    SELECT
        d.campaign_id,
        d.campaign_name,
        d.channel,
        d.region,
        d.product_category,
        ROUND(f.spend_usd, 2) AS spend_usd,
        ROUND(f.revenue_usd, 2) AS revenue_usd,
        ROUND(m.roas, 2) AS roas,
        ROUND(m.cpa, 2) AS cost_per_conversion_usd,
        ROUND(m.ctr * 100, 2) AS ctr_pct
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    JOIN main.campaign_metrics AS m
        ON f.campaign_id = m.campaign_id
    WHERE m.roas < 1.66
      AND f.spend_usd >= 10000
    ORDER BY
        m.roas ASC,
        f.spend_usd DESC,
        d.campaign_id
    LIMIT 10
""").fetchdf()

display(low_roas_df.style.hide(axis="index"))

campaign_id,campaign_name,channel,region,product_category,spend_usd,revenue_usd,roas,cost_per_conversion_usd,ctr_pct
2024_0401,Campaign_2024_0401,Display,Asia,Home,43530.410000,1148.920000,0.030000,4.740000,36.670000
2025_0416,Campaign_2025_0416,Email,Europe,Home,43548.630000,1794.770000,0.040000,1.910000,79.730000
2025_0468,Campaign_2025_0468,Search,South America,Clothing,41768.450000,1937.570000,0.050000,1.290000,98.280000
2024_0142,Campaign_2024_0142,Email,North America,Clothing,33050.910000,1773.520000,0.050000,1.370000,41.340000
2025_0109,Campaign_2025_0109,Email,South America,Home,48699.420000,3095.380000,0.060000,20.000000,12.680000
2025_0013,Campaign_2025_0013,Print,Europe,Services,26160.120000,1875.520000,0.070000,0.700000,86.030000
2024_0120,Campaign_2024_0120,Search,Europe,Electronics,19574.730000,1700.680000,0.090000,0.400000,67.470000
2025_0313,Campaign_2025_0313,Print,Asia,Home,32925.170000,2892.540000,0.090000,0.690000,91.180000
2025_0320,Campaign_2025_0320,Print,Africa,Home,43830.810000,4036.090000,0.090000,1.290000,62.080000
2024_0158,Campaign_2024_0158,Email,Asia,Home,30507.290000,2913.440000,0.100000,1.150000,29.690000


### 7.1 Results and Interpretation

The ten selected campaigns have reported ROAS values ranging from **0.03x to 0.10x**, indicating very low attributed revenue relative to advertising spend under the current metric definitions.

Campaign `2024_0401` ranks first for review, with **$43,530.41 in spend** and only **$1,148.92 in reported revenue**, producing a ROAS of 0.03x. Campaign `2025_0109` has the highest spend among the ten candidates at **$48,699.42**, with reported ROAS of 0.06x.

These campaigns are candidates for investigation, not automatic cancellation recommendations. Before drawing business conclusions, validate the underlying measure assignments and metric calculations, particularly because some campaign-level CTR and ROAS values are unusually extreme compared with the aggregate results.

---

# 8. Campaign Characteristics

## 8.1 Analytical Objective

This section examines campaign performance across four characteristics: device, product category, region, and target audience.

The analysis groups campaign records by each characteristic and aggregates the relevant performance measures. This helps describe how campaign activity, advertising expenditure, and attributed revenue vary across different segments of the dataset.

## 8.2 Dimensions of Analysis

The analysis evaluates four campaign characteristics:

- **Device:** Compares recorded performance across desktop, mobile, and tablet campaigns.
- **Product Category:** Examines differences across the product categories represented in the dataset.
- **Region:** Compares campaign performance across geographic regions.
- **Target Audience:** Examines differences across the recorded audience groups.

These groupings provide complementary perspectives on the campaign portfolio. They are analyzed independently rather than combined into a more granular multidimensional model.

## 8.3 SQL Implementation

The corresponding SQL query uses a common table expression (CTE) and `UNION ALL` to organize the four characteristic-level comparisons into a consistent result structure.

This approach allows related segment analyses to be presented through a common set of output columns while retaining a label identifying the characteristic being examined.

The aggregation summarizes campaign measures within each segment. The query is intended for descriptive comparison, not for isolating the independent effect of any particular characteristic.

## 8.4 Interpretation Approach

Segment-level results should be interpreted using both performance totals and efficiency metrics where available.

Differences in revenue, conversions, or ROAS may reflect variations in campaign volume, expenditure, campaign mix, or other factors not isolated by this analysis. A segment with higher total revenue may simply contain more campaigns or account for more advertising spend.

The results can help identify patterns worth investigating, but they do not establish that a device, product category, region, or audience group directly caused a particular outcome.

## 8.5 Limitations

The analysis is constrained by the dimensions and measures available in the source dataset. It does not control for all possible confounding factors, and the comparisons should not be interpreted as causal or as evidence of statistical significance.

Further analysis could investigate interactions between characteristics or incorporate additional business context if supported by suitable data and a clear analytical requirement.

The corresponding SQL implementation is maintained in `analysis/business_analysis.sql` under **Performance by Campaign Characteristics**.

In [21]:
# 8. LOW-ROAS PERFORMANCE BY CHANNEL

low_roas_channel_df = conn.execute("""
    SELECT
        d.channel,
        COUNT(*) AS campaign_count,
        ROUND(SUM(f.spend_usd), 2) AS total_spend_usd,
        ROUND(SUM(f.revenue_usd), 2) AS total_revenue_usd,
        ROUND(
            SUM(f.revenue_usd) / NULLIF(SUM(f.spend_usd), 0),
            2
        ) AS aggregate_roas,
        ROUND(AVG(m.roas), 2) AS avg_campaign_roas
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    JOIN main.campaign_metrics AS m
        ON f.campaign_id = m.campaign_id
    WHERE m.roas < 1.66
      AND f.spend_usd >= 10000
    GROUP BY d.channel
    ORDER BY
        aggregate_roas ASC,
        total_spend_usd DESC
""").fetchdf()

display(low_roas_channel_df)

,channel,campaign_count,total_spend_usd,total_revenue_usd,aggregate_roas,avg_campaign_roas
0,Email,90,3038168.59,2877955.60,0.95,0.96
1,Display,74,2454234.09,2475483.59,1.01,1.05
2,Search,80,2682342.60,2811288.00,1.05,1.08
3,Social,77,2408680.24,2558986.75,1.06,1.07
4,Print,91,3132794.95,3387562.15,1.08,1.10


### 8.1 Results and Interpretation

Within the screened low-ROAS subset, **Email has the lowest aggregate ROAS at 0.95x**, followed by Display at 1.01x. Print has the highest aggregate ROAS among these channels at 1.08x, but it also has the largest campaign count (91) and the highest total spend ($3.13 million).

Email accounts for $3.04 million in spend and $2.88 million in revenue, resulting in aggregate ROAS below 1.00x. Under the reported metrics, this means attributed revenue is lower than advertising spend for this subset.

Aggregate ROAS is lower than average campaign ROAS for every channel, showing that the two measures produce slightly different views of performance.

These findings apply only to campaigns with ROAS below 1.66x and spend of at least $10,000. They should not be interpreted as rankings of overall channel performance or used alone to determine budget cuts.

---

# 9. Findings & Recommendations

## 9.1 Analytical Summary

The preceding analyses establish a framework for evaluating campaign performance across the marketing portfolio. They examine overall results, channel-level differences, campaign rankings, ROAS distribution, advertising efficiency, and performance across selected campaign characteristics.

The analysis distinguishes between absolute performance, cost efficiency, and return metrics. This distinction is important because a campaign or channel that generates the most revenue does not necessarily deliver the highest return relative to advertising spend.

## 9.2 Evidence-Based Findings

Final findings should be recorded only after the SQL queries have been executed and their outputs verified.

For each finding, document:

1. **Observation:** What the query results show, supported by the relevant metric or comparison.
2. **Evidence:** The values, ranking, or distribution that supports the observation.
3. **Business implication:** Why the observation may matter to marketing performance.
4. **Limitation:** What the analysis cannot establish from the available data.

The following areas provide a structure for documenting the results:

- **Overall performance:** Summarize the portfolio's aggregate campaign activity, spend, revenue, CTR, and ROAS.
- **Channel performance:** Identify meaningful differences in campaign volume, engagement, revenue, and efficiency.
- **Campaign leaders:** Compare the largest revenue contributors with the highest-ROAS campaigns meeting the minimum-spend criterion.
- **Performance distribution:** Assess the distribution of ROAS and identify campaigns that meet the investigation criteria.
- **Marketing efficiency:** Compare CPC, CPM, ROI ratio, and ROAS across channels.
- **Campaign characteristics:** Describe observed differences across device, product category, region, and target audience.

## 9.3 Recommendations Framework

Recommendations should follow from verified findings rather than assumptions about which campaigns or channels perform best.

Potential actions include:

- **Investigate low-ROAS campaigns:** Review campaigns meeting the screening criteria to understand their spend, revenue attribution, and business objectives before considering budget changes.
- **Evaluate high-revenue campaigns:** Examine whether large revenue contributions are accompanied by acceptable advertising efficiency.
- **Review channel efficiency:** Compare CPC, CPM, and ROAS together before identifying potential opportunities for further testing.
- **Investigate segment differences:** Use device, product category, region, and audience comparisons to formulate follow-up questions.
- **Improve measurement quality:** Validate the underlying campaign measures and attribution assumptions before using the results for consequential budget decisions.

These are investigation priorities, not conclusions that a particular campaign, channel, or segment should automatically receive more or less investment.

## 9.4 Decision Boundaries

This analysis is descriptive. The observed relationships do not independently establish causality, incremental revenue, statistical significance, or net profitability.

The ROAS threshold of 1.66 and the minimum advertising spend of $10,000 are screening criteria used by the implemented SQL. They should not be treated as approved business rules without further business validation.

Any budget reallocation should consider campaign objectives, measurement quality, business costs beyond advertising spend, and additional evidence such as controlled experiments where appropriate.

In [22]:
# 9. MARKETING EFFICIENCY BY CHANNEL

channel_efficiency_df = conn.execute("""
    SELECT
        d.channel,
        COUNT(*) AS campaign_count,
        ROUND(
            SUM(f.spend_usd) / NULLIF(SUM(f.clicks), 0),
            2
        ) AS cpc_usd,
        ROUND(
            SUM(f.spend_usd) / NULLIF(SUM(f.impressions), 0) * 1000,
            2
        ) AS cpm_usd,
        ROUND(
            (SUM(f.revenue_usd) - SUM(f.spend_usd))
            / NULLIF(SUM(f.spend_usd), 0),
            4
        ) AS roi_ratio,
        ROUND(
            SUM(f.revenue_usd) / NULLIF(SUM(f.spend_usd), 0),
            2
        ) AS roas
    FROM main.dim_campaign AS d
    JOIN main.fct_campaign_performance AS f
        ON d.campaign_id = f.campaign_id
    GROUP BY d.channel
    ORDER BY
        roas DESC,
        cpc_usd ASC
""").fetchdf()

display(channel_efficiency_df)

,channel,campaign_count,cpc_usd,cpm_usd,roi_ratio,roas
0,Social,358,0.84,279.03,1.7479,2.75
1,Search,349,0.85,308.12,1.7469,2.75
2,Print,386,0.86,297.26,1.6783,2.68
3,Email,389,0.93,310.86,1.6703,2.67
4,Display,349,1.06,308.85,1.6092,2.61


### 9.1 Results and Interpretation

Social has the lowest CPC at **$0.84 per click**, closely followed by Search at $0.85. Social also has the highest ROAS at **2.75x**, tied with Search at the displayed precision.

Social's CPM is $279.03, the lowest among the five channels. Email has the highest CPM at $310.86, while Display has the highest CPC at $1.06 and the lowest ROAS at 2.61x.

All five channels have positive advertising-spend ROI ratios, ranging from 1.6092 to 1.7479. This means reported revenue exceeds advertising spend for each channel, before accounting for other business costs.

Based on these aggregate metrics, Social and Search show the strongest combination of click cost and ROAS, while Display has the weakest relative efficiency. These findings describe the current dataset and should be validated against the underlying measures before informing budget decisions.

---

In [23]:
# 10. PERFORMANCE BY CAMPAIGN CHARACTERISTICS

characteristic_performance_df = conn.execute("""
    WITH characteristic_performance AS (

        SELECT
            'Device' AS characteristic,
            d.device AS category,
            COUNT(*) AS campaign_count,
            SUM(f.spend_usd) AS total_spend_usd,
            SUM(f.revenue_usd) AS total_revenue_usd,
            SUM(f.conversions) AS total_conversions
        FROM main.dim_campaign AS d
        JOIN main.fct_campaign_performance AS f
            ON d.campaign_id = f.campaign_id
        GROUP BY d.device

        UNION ALL

        SELECT
            'Product Category' AS characteristic,
            d.product_category AS category,
            COUNT(*) AS campaign_count,
            SUM(f.spend_usd) AS total_spend_usd,
            SUM(f.revenue_usd) AS total_revenue_usd,
            SUM(f.conversions) AS total_conversions
        FROM main.dim_campaign AS d
        JOIN main.fct_campaign_performance AS f
            ON d.campaign_id = f.campaign_id
        GROUP BY d.product_category

        UNION ALL

        SELECT
            'Region' AS characteristic,
            d.region AS category,
            COUNT(*) AS campaign_count,
            SUM(f.spend_usd) AS total_spend_usd,
            SUM(f.revenue_usd) AS total_revenue_usd,
            SUM(f.conversions) AS total_conversions
        FROM main.dim_campaign AS d
        JOIN main.fct_campaign_performance AS f
            ON d.campaign_id = f.campaign_id
        GROUP BY d.region

        UNION ALL

        SELECT
            'Target Audience' AS characteristic,
            d.target_audience AS category,
            COUNT(*) AS campaign_count,
            SUM(f.spend_usd) AS total_spend_usd,
            SUM(f.revenue_usd) AS total_revenue_usd,
            SUM(f.conversions) AS total_conversions
        FROM main.dim_campaign AS d
        JOIN main.fct_campaign_performance AS f
            ON d.campaign_id = f.campaign_id
        GROUP BY d.target_audience
    )

    SELECT
        characteristic,
        category,
        campaign_count,
        ROUND(total_spend_usd, 2) AS total_spend_usd,
        ROUND(total_revenue_usd, 2) AS total_revenue_usd,
        ROUND(
            total_revenue_usd / NULLIF(total_spend_usd, 0),
            2
        ) AS aggregate_roas,
        ROUND(
            total_spend_usd / NULLIF(total_conversions, 0),
            2
        ) AS cost_per_conversion_usd
    FROM characteristic_performance
    ORDER BY
        characteristic,
        aggregate_roas DESC,
        category
""").fetchdf()

display(characteristic_performance_df)

,characteristic,category,campaign_count,total_spend_usd,total_revenue_usd,aggregate_roas,cost_per_conversion_usd
0,Device,Tablet,577,15100128.14,42426350.30,2.81,2.07
1,Device,Desktop,607,15512820.72,41901081.26,2.70,2.03
2,Device,Mobile,647,16505978.51,42399996.27,2.57,1.88
3,Product Category,Services,172,4107265.80,13664645.48,3.33,1.01
4,Product Category,Travel,223,5691585.78,16511011.10,2.90,1.14
5,Product Category,Home,376,9544613.16,25771666.57,2.70,1.84
6,Product Category,Clothing,351,9168964.35,24433340.03,2.66,1.96
7,Product Category,Electronics,339,9042549.28,23074842.39,2.55,2.10
8,Product Category,Health,175,4372321.33,11044271.17,2.53,20.75
9,Product Category,Automotive,195,5191627.67,12227651.09,2.36,19.85


### 9.1 Results and Interpretation

**Device:** Tablet has the highest aggregate ROAS at 2.81x, followed by Desktop at 2.70x and Mobile at 2.57x. Mobile has the lowest cost per conversion at $1.88, showing that the strongest ROAS and lowest conversion cost do not necessarily occur in the same category.

**Product category:** Services has the highest aggregate ROAS at 3.33x, followed by Travel at 2.90x. Automotive has the lowest aggregate ROAS at 2.36x. The reported cost per conversion for Health ($20.75) and Automotive ($19.85) is notably higher than for the other categories and should be validated.

**Region:** South America has the highest aggregate ROAS at 2.86x, while North America has the lowest at 2.55x. Africa has the highest total spend among the regions, at approximately $10.03 million.

**Target audience:** Adults has the highest aggregate ROAS at 2.73x, followed by Seniors at 2.68x and Youth at 2.66x. The differences are relatively small.

These are descriptive comparisons of the observed dataset. They do not establish that a device, product category, region, or audience causes better performance. Validate the underlying measures and derived metrics before using these findings for business decisions.

---

# 10. Limitations & Reproducibility

## 10.1 Data and Measurement Limitations

The analysis uses three marketing campaign CSV files containing 1,831 records and 15 columns in total. The dataset provides campaign attributes, engagement measures, conversions, advertising spend, and attributed revenue.

The results depend on the accuracy and interpretation of these underlying fields. In particular, the source measures for impressions, clicks, and conversions were reconstructed upstream by ordering the three original measure values from largest to smallest. Although this transformation has a corresponding dbt unit test, the test verifies the implemented reassignment logic rather than independently establishing the semantic correctness of the reconstructed measures.

Consequently, conclusions involving engagement rates, conversion performance, and related metrics should be interpreted with this limitation in mind.

## 10.2 Analytical Limitations

The analysis has several additional limitations:

- **Descriptive analysis:** Grouped comparisons identify observed patterns but do not establish causality.
- **Revenue attribution:** Attributed revenue may not represent incremental revenue generated by advertising.
- **Advertising-spend-only ROI:** The ROI ratio excludes other operating and business costs, so it is not a complete net-profit measure.
- **Screening thresholds:** The ROAS threshold of 1.66 and minimum spend of $10,000 are analytical criteria, not formally approved business rules.
- **Available dimensions:** The analysis is limited to the campaign characteristics represented in the source data.
- **Data scope:** Results describe the available dataset and should not automatically be generalized to other periods or marketing portfolios.

## 10.3 Database and Execution Dependencies

The analytical queries in `analysis/business_analysis.sql` reference PostgreSQL serving-layer objects:

- `analytics.campaign_metrics`
- `analytics.dim_campaign`
- `analytics.fct_campaign_performance`

The project's dbt development workflow uses DuckDB, while PostgreSQL serves a separate analytical role. Building the dbt models in DuckDB does not, by itself, demonstrate that the PostgreSQL serving tables have been refreshed.

Successful execution therefore depends on the required PostgreSQL objects being available and populated with the intended data.

## 10.4 Reproducibility Workflow

A reproducible execution should follow this sequence:

1. **Validate source data:** Confirm the expected input files and source-data scope.
2. **Run ingestion and transformation:** Execute the established Python ingestion and dbt workflow.
3. **Validate model quality:** Run the applicable dbt tests and review their results.
4. **Refresh the serving layer:** Ensure the PostgreSQL analytical objects contain the intended version of the transformed data.
5. **Execute the analytical SQL:** Run the queries in `analysis/business_analysis.sql` against the appropriate PostgreSQL database.
6. **Verify the outputs:** Confirm the result values, metric calculations, and reporting scope before including findings in this notebook.
7. **Document the results:** Record verified observations, business implications, and limitations.

The notebook should not be considered fully reproducible merely because its SQL source is available. Reproducibility also requires the appropriate dependencies, database objects, data state, and execution instructions.

## 10.5 Portfolio and Maintenance Considerations

The analytical SQL remains the reference implementation for the business queries, while this notebook documents their purpose, interpretation, and limitations.

Future improvements may include stronger validation of source measure semantics, automated reconciliation between the DuckDB transformation layer and PostgreSQL serving tables, and more formalized business definitions for performance thresholds.

These improvements should be introduced when justified by business requirements rather than adding complexity without a clear analytical purpose.